In [1]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 71.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 78.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.3/121.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 969.1/969.1 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 11.7 MB/s eta 0:00:00


In [25]:
!pip install boto3
!pip install awscli

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 57.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 81.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 29.8 MB/s eta 0:00:00
  Attempting uninstall: rsa
    Found existing installation: rsa 4.9.1
    Uninstalling rsa-4.9.1:
      Successfully uninstalled rsa-4.9.1
  Attempting uninstall: docutils
    Found existing installation: docutils 0.21.2
    Uninstalling docutils-0.21.2:
      Successfully uninstalled docutils-0.21.2
  Attempting uninstall: botocore
    Found existing installation: botocore 1.43.36
    Uninstalling botocore-1.43.36:
      Successfully uninstalled botocore-1.43.36
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
sphinx 8.

In [2]:
import mlflow
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow.lightgbm
import lightgbm as lgb
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer

In [3]:
data = pd.read_csv("/kaggle/input/datasets/bjdhdhjdbd/twitter-data-sentiment/clean_data.csv")
data.head()

,clean_text,category
0,family mormon never try explain still stare pu...,1.0
1,buddhism much lot compatible christianity espe...,1.0
2,seriously say thing first get complex explain ...,-1.0
3,learn want teach different focus goal not wrap...,0.0
4,benefit may want read live buddha live christ ...,1.0


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199702 entries, 0 to 199701
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   clean_text  199508 non-null  object 
 1   category    199702 non-null  float64
dtypes: float64(1), object(1)
memory usage: 3.0+ MB


In [5]:
data = data.dropna()

In [7]:
data["clean_text"].isna().sum()

np.int64(0)

In [8]:
vectorizer = CountVectorizer(max_features=1000)

In [9]:

X = vectorizer.fit_transform(data['clean_text']).toarray()

In [10]:
y = data["category"]

In [13]:
x_train , x_test, y_train  , y_test = train_test_split(X , y , test_size=0.2, random_state=42)
mapping = {
    -1: 0,
     0: 1,
     1: 2
}
y_train = y_train.map(mapping)
y_test = y_test.map(mapping)

In [15]:
model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,
    random_state=42
)

In [16]:
search_space = {
    "num_leaves": Integer(20, 200),
    "max_depth": Integer(-1, 15),
    "learning_rate": Real(0.01, 0.3),
    "n_estimators": Integer(100, 800),
    "min_child_samples": Integer(5, 100),
    "subsample": Real(0.5, 1.0),
    "colsample_bytree": Real(0.5, 1.0),
    "reg_alpha": Real(0.0, 5.0),
    "reg_lambda": Real(0.0, 5.0)
}

In [17]:
bayes_search = BayesSearchCV(

    estimator=model,

    search_spaces=search_space,

    n_iter=5,

    cv=5,

    scoring="accuracy",

    n_jobs=-1,

    random_state=42,

    verbose=2
)

In [18]:
bayes_search.fit(x_train, y_train)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.824386 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5148
[LightGBM] [Info] Number of data points in the train set: 127684, number of used features: 1000
[LightGBM] [Info] Start training from score -1.514750
[LightGBM] [Info] Start training from score -1.081106
[LightGBM] [Info] Start training from score -0.818899
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.636901 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5365
[LightGBM] [Info] Number of data points in the train set: 159606, number of used features: 999
[LightGBM] [Info] Start training from score -1.514735
[LightGBM] [Info] Start training from score -1.081117
[LightGBM] [Info] Start training from score -0.818899
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

BayesSearchCV(cv=5,
              estimator=LGBMClassifier(num_class=3, objective='multiclass',
                                       random_state=42),
              n_iter=5, n_jobs=-1, random_state=42, scoring='accuracy',
              search_spaces={'colsample_bytree': Real(low=0.5, high=1.0, prior='uniform', transform='normalize'),
                             'learning_rate': Real(low=0.01, high=0.3, prior='uniform', transform='normalize'),
                             'max_depth': Integer(low=-1,...
                             'n_estimators': Integer(low=100, high=800, prior='uniform', transform='normalize'),
                             'num_leaves': Integer(low=20, high=200, prior='uniform', transform='normalize'),
                             'reg_alpha': Real(low=0.0, high=5.0, prior='uniform', transform='normalize'),
                             'reg_lambda': Real(low=0.0, high=5.0, prior='uniform', transform='normalize'),
                             'subsample': Real(low=0.5, high=1.0, prior='uniform', transform='normalize')},
              verbose=2)

In [19]:
params = bayes_search.best_params_
params

OrderedDict([('colsample_bytree', 0.8997767208035865),
             ('learning_rate', 0.13702846406786778),
             ('max_depth', 7),
             ('min_child_samples', 73),
             ('n_estimators', 733),
             ('num_leaves', 149),
             ('reg_alpha', 2.1208903623034105),
             ('reg_lambda', 3.2514205087388133),
             ('subsample', 0.6765419227639857)])

In [20]:
model = bayes_search.best_estimator_
model

LGBMClassifier(colsample_bytree=0.8997767208035865,
               learning_rate=0.13702846406786778, max_depth=7,
               min_child_samples=73, n_estimators=733, num_class=3,
               num_leaves=149, objective='multiclass', random_state=42,
               reg_alpha=2.1208903623034105, reg_lambda=3.2514205087388133,
               subsample=0.6765419227639857)

In [21]:
y_pred = model.predict(x_test)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [22]:
rev_mapping = {
    0: -1,
    1: 0,
    2: 1
}

y_pred = np.vectorize(rev_mapping.get)(y_pred)
y_test = np.vectorize(rev_mapping.get)(y_test)

In [23]:

classification_rep = classification_report(
    y_test,
    y_pred,
    output_dict=True
)
print(classification_rep)


{'-1': {'precision': 0.7842486421243211, 'recall': 0.6002309468822171, 'f1-score': 0.6800104657247514, 'support': 8660.0}, '0': {'precision': 0.7279043280182232, 'recall': 0.9378531073446328, 'f1-score': 0.8196479528038731, 'support': 13629.0}, '1': {'precision': 0.8697976326842306, 'recall': 0.7760177141883836, 'f1-score': 0.820235844810514, 'support': 17613.0}, 'accuracy': 0.793143200842063, 'macro avg': {'precision': 0.7939835342755917, 'recall': 0.7713672561384112, 'f1-score': 0.7732980877797129, 'support': 39902.0}, 'weighted avg': {'precision': 0.8027654511509282, 'recall': 0.793143200842063, 'f1-score': 0.7896016870479654, 'support': 39902.0}}


In [24]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7931

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightG

In [27]:
mlflow.set_tracking_uri("http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/")
mlflow.set_experiment("LGBM base_line ")

2026/07/26 18:32:12 INFO mlflow.tracking.fluent: Experiment with name 'LGBM base_line ' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://zg-mlflow/2', creation_time=1785090732331, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1785090732331, lifecycle_stage='active', name='LGBM base_line ', tags={}, trace_location=None, workspace='default'>

In [29]:

try:
    with mlflow.start_run() as run:


        mlflow.set_tag("model", "LGBM")
        mlflow.set_tag("optimization", "Bayesian Search")


        mlflow.log_param("vectorizer_type", "CountVectorizer")
        mlflow.log_param("vectorizer_max_features", 1000)
        mlflow.log_params(params)


        mlflow.log_metric("accuracy", accuracy)

        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric_name, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric_name}", value)


        mlflow.lightgbm.log_model(
            lgb_model=model,
           artifact_path="lightgbm_model"
           )

        data.to_csv("dataset.csv", index=False)
        mlflow.log_artifact("dataset.csv")

        # Confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title("Confusion Matrix")

        plt.savefig("confusion_matrix.png")
        plt.close()

        mlflow.log_artifact("confusion_matrix.png")

        print(f"Run ID: {run.info.run_id}")

except Exception as e:
    print(f"MLflow logging failed: {e}")

finally:
    
    import os

    if os.path.exists("dataset.csv"):
        os.remove("dataset.csv")

    if os.path.exists("confusion_matrix.png"):
        os.remove("confusion_matrix.png")

2026/07/26 18:35:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run ID: 233f38babe2542fb8661dcb23d0c8aec
🏃 View run mysterious-conch-151 at: http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/#/experiments/2/runs/233f38babe2542fb8661dcb23d0c8aec
🧪 View experiment at: http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/#/experiments/2
